In [33]:
import re, csv, math
import pandas as pd
from pathlib import Path

path_cat = Path("cat_colonias.sql")
path_edi = Path("edificios_escolares.sql")

def split_tuples(values_blob: str):
    tuples = []
    in_str = False
    esc = False
    depth = 0
    start = None

    for i, ch in enumerate(values_blob):
        if ch == "\\" and in_str:
            esc = not esc
            continue
        if ch == "'" and not esc:
            in_str = not in_str
        esc = False

        if not in_str:
            if ch == "(":
                if depth == 0:
                    start = i
                depth += 1
            elif ch == ")":
                depth -= 1
                if depth == 0 and start is not None:
                    tuples.append(values_blob[start:i+1])
                    start = None
    return tuples

def parse_tuple(tuple_str: str):
    inner = tuple_str.strip()[1:-1]
    row = next(csv.reader([inner], delimiter=",", quotechar="'", escapechar="\\"))
    return [x.strip() for x in row]

def extract_cat_colonias_cp_lat_lon(sql_path: Path):
    text = sql_path.read_text(encoding="utf-8", errors="ignore")
    insert_re = re.compile(r"INSERT\s+INTO\s+`?cat_colonias`?\s+VALUES\s*(.*?);", re.IGNORECASE | re.DOTALL)

    out = []
    for m in insert_re.finditer(text):
        for t in split_tuples(m.group(1)):
            row = parse_tuple(t)
            if len(row) < 4:
                continue
            cp  = row[1]          # 2do valor = CP (según tu INSERT)
            lat = row[-2]         # penúltimo = lat
            lon = row[-1]         # último = lon
            out.append((cp, lat, lon))

    df = pd.DataFrame(out, columns=["codigo_postal", "latitud", "longitud"])

    df["codigo_postal"] = (
        df["codigo_postal"].astype(str).str.strip()
        .str.replace(".0", "", regex=False)
        .str.replace(r"\D", "", regex=True)
        .str.zfill(5)
    )
    df["latitud"]  = pd.to_numeric(df["latitud"], errors="coerce")
    df["longitud"] = pd.to_numeric(df["longitud"], errors="coerce")

    df = df.dropna(subset=["codigo_postal","latitud","longitud"])
    df = df[df["latitud"].between(-90, 90) & df["longitud"].between(-180, 180)]
    return df

def extract_edificios_cp_lat_lon(sql_path: Path):
    text = sql_path.read_text(encoding="utf-8", errors="ignore")
    insert_re = re.compile(r"INSERT\s+INTO\s+`?edificios_escolares`?\s+VALUES\s*(.*?);", re.IGNORECASE | re.DOTALL)

    out = []
    for m in insert_re.finditer(text):
        for t in split_tuples(m.group(1)):
            row = parse_tuple(t)
            if len(row) < 10:
                continue
            # En tu INSERT, CP está a -8 desde el final:
            # ... id_asentamiento, CP, nombre_colonia, tipo_id, tipo, num_ext, num_int, lat, lon
            cp  = row[-8]
            lat = row[-2]
            lon = row[-1]
            out.append((cp, lat, lon))

    df = pd.DataFrame(out, columns=["codigo_postal", "latitud", "longitud"])

    df["codigo_postal"] = (
        df["codigo_postal"].astype(str).str.strip()
        .str.replace(".0", "", regex=False)
        .str.replace(r"\D", "", regex=True)
        .str.zfill(5)
    )
    df["latitud"]  = pd.to_numeric(df["latitud"], errors="coerce")
    df["longitud"] = pd.to_numeric(df["longitud"], errors="coerce")

    df = df.dropna(subset=["codigo_postal","latitud","longitud"])
    df = df[df["latitud"].between(-90, 90) & df["longitud"].between(-180, 180)]
    return df


In [37]:
from pathlib import Path

path_cat = Path("cat_colonias.sql")
path_edi = Path("edificios_escolares.sql")

print("cat exists:", path_cat.exists(), "size:", path_cat.stat().st_size)
print("edi exists:", path_edi.exists(), "size:", path_edi.stat().st_size)

print("\n--- Primeras 5 líneas cat_colonias.sql ---")
with open(path_cat, "r", encoding="utf-8", errors="ignore") as f:
    for _ in range(5):
        print(f.readline().rstrip("\n"))

print("\n--- Primeras 5 líneas edificios_escolares.sql ---")
with open(path_edi, "r", encoding="utf-8", errors="ignore") as f:
    for _ in range(5):
        print(f.readline().rstrip("\n"))


cat exists: True size: 1047820
edi exists: True size: 7751472

--- Primeras 5 líneas cat_colonias.sql ---
/*
 Navicat Premium Data Transfer

 Source Server         : Local_Server
 Source Server Type    : MySQL

--- Primeras 5 líneas edificios_escolares.sql ---
/*
 Navicat Premium Data Transfer

 Source Server         : Local_Server
 Source Server Type    : MySQL


In [38]:
text_cat = path_cat.read_text(encoding="utf-8", errors="ignore")
text_edi = path_edi.read_text(encoding="utf-8", errors="ignore")

print("cat contiene 'INSERT INTO':", "INSERT INTO" in text_cat.upper())
print("edi contiene 'INSERT INTO':", "INSERT INTO" in text_edi.upper())

print("cat contiene 'cat_colonias':", "cat_colonias" in text_cat)
print("edi contiene 'edificios_escolares':", "edificios_escolares" in text_edi)


cat contiene 'INSERT INTO': True
edi contiene 'INSERT INTO': True
cat contiene 'cat_colonias': True
edi contiene 'edificios_escolares': True


In [39]:
import re

count_cat = len(re.findall(r"INSERT\s+INTO", text_cat, re.IGNORECASE))
count_edi = len(re.findall(r"INSERT\s+INTO", text_edi, re.IGNORECASE))

print("INSERTs en cat_colonias.sql:", count_cat)
print("INSERTs en edificios_escolares.sql:", count_edi)


INSERTs en cat_colonias.sql: 5757
INSERTs en edificios_escolares.sql: 13865


In [40]:
m = re.search(r"(INSERT\s+INTO\s+.*?;)", text_cat, re.IGNORECASE | re.DOTALL)
print("Primer INSERT en cat:", (m.group(1)[:600] if m else "NO ENCONTRÉ"))

m2 = re.search(r"(INSERT\s+INTO\s+.*?;)", text_edi, re.IGNORECASE | re.DOTALL)
print("\nPrimer INSERT en edi:", (m2.group(1)[:600] if m2 else "NO ENCONTRÉ"))


Primer INSERT en cat: INSERT INTO `cat_colonias` VALUES (1, '25000', 'Saltillo Zona Centro', 'Colonia', 'Saltillo', 'Coahuila de Zaragoza', 'Saltillo', '030', '1', '0001', '25.42013552', '-100.9976232');

Primer INSERT en edi: INSERT INTO `edificios_escolares` VALUES ('05ABJ0001Z', '6', 'ADMINISTRATIVO', '400', 'DISCONTINUO', '32', 'REPRESENTAR A LA COORDINACIÓN NACIONAL DE BECAS PARA EL BIENESTAR BENITO JUAREZ EN LA ENTI ', '11', 'FEDERAL', '00', 'NO APLICA', '000', 'NO APLICA', 'OFICINA DE REPRESENTACIÓN DE LA CNBBBJ EN EL ESTADO DE COAHUILA', '1', 'ALTA', '999', 'NO APLICA', '2020-01-08 13:00:22', NULL, NULL, NULL, NULL, '500', 'LAGUNA', '035', 'TORREON', '0001', 'TORREÓN', '1', 'URBANO', 2535, '27000', 'TORREÓN CENTRO', '7', 'COLONIA', 'SN', '', '25.538756', '-103.453245');


In [34]:
df_col = extract_cat_colonias_cp_lat_lon(path_cat)
df_edi = extract_edificios_cp_lat_lon(path_edi)

print("Colonias:", df_col.shape)
print(df_col.head())

print("\nEdificios:", df_edi.shape)
print(df_edi.head())

common = set(df_col["codigo_postal"]).intersection(set(df_edi["codigo_postal"]))
print("\nCP en común:", len(common))
print("Ejemplos:", list(common)[:20])


Colonias: (0, 3)
Empty DataFrame
Columns: [codigo_postal, latitud, longitud]
Index: []

Edificios: (0, 3)
Empty DataFrame
Columns: [codigo_postal, latitud, longitud]
Index: []

CP en común: 0
Ejemplos: []


In [35]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1)*math.cos(lat2)*math.sin(dlon/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    return R * c

cp_coords = (
    df_col.groupby("codigo_postal", as_index=False)
          .agg(lat_cp=("latitud","mean"), lon_cp=("longitud","mean"))
)

df = df_edi.merge(cp_coords, on="codigo_postal", how="inner")

df["distancia_km"] = df.apply(
    lambda r: haversine_km(r["latitud"], r["longitud"], r["lat_cp"], r["lon_cp"]),
    axis=1
)

print("Total con CP match:", len(df))
df.head()


Total con CP match: 0


,codigo_postal,latitud,longitud,lat_cp,lon_cp,distancia_km


In [36]:
df_10 = df[df["distancia_km"] >= 10].sort_values("distancia_km", ascending=False)
print("Total >= 10 km:", len(df_10))

df_10.to_csv("edificios_10km_o_mas.csv", index=False, encoding="utf-8-sig")
df_10.head(20)


Total >= 10 km: 0


,codigo_postal,latitud,longitud,lat_cp,lon_cp,distancia_km
